# P5: Swarm Coherence – Φ-loven i sværm

**Trykk `Runtime → Run all`**

100 agenter. Ingen sentral kontroll. Kun Φ-loven.
Resonans oppstår når >80% er i koherens-sonen [1888, 4766] samtidig.

In [ ]:
!pip install -q numpy matplotlib tqdm

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

class Agent:
    def __init__(self, agent_id, initial_tau=4000.0):
        self.id = agent_id
        self.tau = initial_tau
        self.C_0 = 4495.27
        self.tau_min = 1888.0
        self.tau_max = 4766.0
        self.energy = 1.0
        self.coherent_steps = 0

    def step(self, noise_level=0.1):
        self.tau += abs(np.random.normal(0, noise_level * 10))
        if self.tau < self.tau_min:
            self.tau += (self.tau_min - self.tau) * 0.1
        elif self.tau > self.tau_max:
            self.tau += (self.tau_max - self.tau) * 0.1
        is_coherent = self.tau_min <= self.tau <= self.tau_max
        if is_coherent:
            self.coherent_steps += 1
            self.energy = min(1.0, self.energy + 0.01)
        else:
            self.coherent_steps = 0
            self.energy = max(0.0, self.energy - 0.05)
        return is_coherent, self.tau

def run_swarm(n_agents=100, n_steps=200, with_phi=True):
    agents = [Agent(i) for i in range(n_agents)]
    tau_matrix = np.zeros((n_steps, n_agents))
    coherence_count = np.zeros(n_steps)
    resonance_events = []
    label = 'Med Φ-lov' if with_phi else 'Uten Φ-lov'
    print(f'\nStarter simulering: {n_agents} agenter, {n_steps} steg — {label}')
    for t in tqdm(range(n_steps)):
        coherent = 0
        for i, agent in enumerate(agents):
            if not with_phi:
                agent.tau += abs(np.random.normal(0, 50))
                is_c = agent.tau_min <= agent.tau <= agent.tau_max
                tau_matrix[t, i] = agent.tau
            else:
                is_c, tau = agent.step()
                tau_matrix[t, i] = tau
            if is_c:
                coherent += 1
        coherence_count[t] = coherent
        if coherent / n_agents > 0.8:
            resonance_events.append(t)
    return {'tau_matrix': tau_matrix, 'coherence_count': coherence_count, 'resonance_events': resonance_events}

N_AGENTS = 100
N_STEPS = 200
hist_with = run_swarm(N_AGENTS, N_STEPS, with_phi=True)
hist_without = run_swarm(N_AGENTS, N_STEPS, with_phi=False)

print(f'\nResonans-hendelser (Med Φ): {len(hist_with["resonance_events"])}')
print(f'Resonans-hendelser (Uten Φ): {len(hist_without["resonance_events"])}')
if hist_with['resonance_events']:
    print('✨ Svermen fant harmoni. Tofoo. Φ 🟢')
else:
    print('Ingen resonans oppnådd.')

In [ ]:
steps = range(N_STEPS)
fig, axs = plt.subplots(3, 1, figsize=(14, 12))

axs[0].plot(steps, hist_with['coherence_count'], color='#2ecc71', linewidth=2, label='Med Φ-lov')
axs[0].plot(steps, hist_without['coherence_count'], color='#e74c3c', linewidth=2, linestyle='--', label='Uten Φ-lov')
axs[0].axhline(N_AGENTS * 0.8, color='blue', linestyle=':', label='Resonans-terskel (80%)')
axs[0].set_title('Antall Agenter i Koherens-sonen', fontsize=13, fontweight='bold')
axs[0].set_ylabel('Antall Agenter'); axs[0].legend(); axs[0].grid(alpha=0.3)

axs[1].plot(steps, np.mean(hist_with['tau_matrix'], axis=1), color='#2ecc71', label='Gj.snitt Tau (Med Φ)')
axs[1].plot(steps, np.mean(hist_without['tau_matrix'], axis=1), color='#e74c3c', label='Gj.snitt Tau (Uten Φ)')
axs[1].axhline(4495.27, color='blue', linestyle=':', label='C₀ = 4495.27')
axs[1].axhspan(1888, 4766, color='#f1c40f', alpha=0.1, label='Koherens-sone')
axs[1].set_title('Svermens Puls (Gjennomsnittlig Tau)', fontsize=13, fontweight='bold')
axs[1].set_ylabel('Tau (bits)'); axs[1].legend(); axs[1].grid(alpha=0.3)

r = hist_with['resonance_events']
if r:
    axs[2].scatter(r, [1]*len(r), color='#f1c40f', s=120, zorder=5, label=f'Resonans ({len(r)} hendelser)')
axs[2].set_title('Resonans-hendelser (>80% i koherens samtidig)', fontsize=13, fontweight='bold')
axs[2].set_xlabel('Tid (Steg)'); axs[2].set_yticks([]); axs[2].legend(); axs[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('p5_swarm_results.png', dpi=300)
plt.show()
print('Ferdig! Tofoo. Φ 🟢')